# Simulating a 3D balanced SSFP acquisition

`01_build.ipynb` wrote two acquisitions and asserted that they are balanced. Balance is a statement
about gradient waveforms, and it is checked there where it is defined. **It is not a statement that
the acquisition produces the image it is supposed to.**

This notebook answers the questions that a waveform check cannot:

```text
1. what preparation should this protocol use?
2. what did 01 actually write?
3. does the continuous acquisition reconstruct into the right volume?
4. what does interrupted segmentation change?
5. what remains a protocol choice?
```

Everything here reads the artifacts `01` wrote. Nothing rebuilds a second, similar acquisition and
checks that instead -- the files simulated below are the files that notebook ships.

In [ ]:
import os
import sys

#: MRzero's inner loop takes every core it is offered.  Hold it to four, before torch is imported.
SIM_THREADS = 4
for _var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'RAYON_NUM_THREADS',
             'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_var] = str(SIM_THREADS)

import contextlib
import os as _os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

import seqcraft as sc

torch.set_num_threads(SIM_THREADS)
plt.rcParams['figure.dpi'] = 72
warnings.filterwarnings('ignore', category=sc.SeqCraftWarning)

sys.path.insert(0, str(Path('..').resolve()))
import phantom as ph            # noqa: E402 -- the shared BrainWeb preparation, one directory up

SEQ_DIR = Path('seq')
#: Diagnostics are not acquisitions: single locations read hundreds of times, preparation pulses
#: probed on their own.  They go in their own directory so `seq/` holds only what 01 ships.
DIAG_DIR = SEQ_DIR / 'diagnostics'
DIAG_DIR.mkdir(parents=True, exist_ok=True)

#: The brain volumes are the expensive part -- two full acquisitions, a few minutes each.  Every
#: *number* below is measured on a single voxel and runs either way.
BRAIN_IMAGES = True

nominal = np.load(SEQ_DIR / 'bssfp_3d_nominal.npz')
NX, NY, NZ = (int(v) for v in nominal['matrix'])
FOV_MM = tuple(float(v) for v in nominal['fov_mm'])
SLAB_MM = float(nominal['slab_mm'])
FLIP_DEG = float(nominal['flip_deg'])
BANDWIDTH_HZ_PX = float(nominal['bandwidth_hz_px'])
TE_S, TR_S = float(nominal['te_s']), float(nominal['tr_s'])
PHASE_STEP = float(nominal['phase_step'])
PREP_PULSE_S = float(nominal['prep_pulse_s'])
SEGMENTS = int(nominal['segments'])
PARTITIONS_PER_SEGMENT = int(nominal['partitions_per_segment'])
RECOVERY_S = float(nominal['recovery_s'])
TABLE = [tuple(int(v) for v in row) for row in nominal['table']]
CENTER_LINE = int(nominal['center_line'])
CENTER_PARTITION = int(nominal['center_partition'])

print(f'{NX} x {NY} x {NZ} over {FOV_MM[0]:.0f} x {FOV_MM[1]:.0f} x {FOV_MM[2]:.0f} mm, '
      f'{SLAB_MM:.0f} mm slab')
print(f'TE {TE_S * 1e3:.3f} ms   TR {TR_S * 1e3:.3f} ms   flip {FLIP_DEG:.0f} deg   '
      f'phase step {PHASE_STEP:.0f} deg')
print(f'preparation {str(nominal["preparation"])}, {SEGMENTS} segments of '
      f'{PARTITIONS_PER_SEGMENT} partitions, {RECOVERY_S * 1e3:.0f} ms recovery')
print(f'{len(TABLE)} imaging repetitions; budget {torch.get_num_threads()} torch threads, '
      f'brain images {"on" if BRAIN_IMAGES else "off"}')

In [ ]:
opts = pp.Opts(
    max_grad=32, grad_unit='mT/m', max_slew=130, slew_unit='T/m/s', B0=3.0,
    rf_dead_time=100e-6, rf_ringdown_time=30e-6, adc_dead_time=10e-6,
)
kernel = sc.modules.bSSFP3DTR(
    opts=opts, fov_mm=FOV_MM, matrix=(NX, NY, NZ), slab_thickness_mm=SLAB_MM,
    flip_deg=FLIP_DEG, bandwidth_hz_px=BANDWIDTH_HZ_PX,
)
#: The kernel rebuilt here must be the one 01 wrote with, or nothing below is about those files.
assert abs(kernel.tr_s - TR_S) < 1e-12 and abs(kernel.te_s - TE_S) < 1e-12
assert (kernel.center_line, kernel.center_partition) == (CENTER_LINE, CENTER_PARTITION)
ECHO = kernel.ro.echo_sample(0)
SAMPLES = int(kernel.ro.adc.num_samples)


@contextlib.contextmanager
def quiet():
    '''Silence the solver's per-call log: it is written to file descriptor 1 from Rust.'''
    saved = _os.dup(1)
    null = _os.open(_os.devnull, _os.O_WRONLY)
    try:
        _os.dup2(null, 1)
        yield
    finally:
        _os.dup2(saved, 1)
        _os.close(null)
        _os.close(saved)


def simulate(path, obj, *, states=200):
    '''Run one written .seq against `obj`; returns one row of samples per repetition.'''
    seq = mr0.Sequence.import_file(str(path))
    with quiet():
        graph = mr0.compute_graph(seq, obj, max_state_count=states, min_state_mag=1e-4)
        signal = mr0.execute_graph(graph, seq, obj, min_emitted_signal=1e-4,
                                   min_latent_signal=1e-5, print_progress=False)
    return np.asarray(signal.detach().cpu().numpy()).reshape(-1, SAMPLES)


def one_voxel(T1, T2, B0=0.0):
    '''A single on-resonance spin, so each repetition returns one complex number.'''
    return mr0.CustomVoxelPhantom(
        pos=[[0.0, 0.0, 0.0]], PD=1.0, T1=T1, T2=T2, T2dash=1e3, D=0.0, B0=B0,
        voxel_size=0.1, voxel_shape='box',
    ).build()


TISSUES = {
    'white matter': dict(T1=0.83, T2=0.080),
    'grey matter':  dict(T1=1.33, T2=0.110),
    'CSF':          dict(T1=4.00, T2=2.000),
}
print(f'echo sample {ECHO} of {SAMPLES}')

## 1. What preparation should this protocol use?

Reaching the bSSFP steady state from equilibrium takes **four to five times T1**, which at a 7 ms TR
is thousands of repetitions. The literature names *catalyzing* schemes to shorten the approach --
the Handbook's **alpha/2 - TR/2**, and linear flip-angle ramps (Le Roux 2003; Hargreaves 2001) --
but it sets **no criterion for choosing between them**. So the choice is made here, by measurement,
and `01` ships the winner.

Each candidate is built as a diagnostic file: the same kernel, held at the **centre of k-space** for
every repetition so the encoding history is identical and only the magnetisation preparation
differs. Every repetition is acquired, which is why they carry a `NAV` label -- they all sit at one
k-space address, so none of them is imaging, and saying so keeps them out of the duplicate-address
check that would otherwise be right to refuse the file.

In [ ]:
NAV = pp.make_label(type='SET', label='NAV', value=1)
N_REPS = 300


def ramp_kernels(n, final_deg=FLIP_DEG):
    '''A linear flip-angle ramp: n repetitions rising to the imaging angle.'''
    return [
        sc.modules.bSSFP3DTR(
            opts=opts, fov_mm=FOV_MM, matrix=(NX, NY, NZ), slab_thickness_mm=SLAB_MM,
            flip_deg=final_deg * (k + 1) / (n + 1), bandwidth_hz_px=BANDWIDTH_HZ_PX, tr_s=TR_S,
        )
        for k in range(n)
    ]


def half_angle_prep(flip_deg=FLIP_DEG):
    '''The alpha/2 pulse of the alpha/2 - TR/2 scheme, as a block of its own.'''
    exc = sc.modules.Excitation(opts=opts, flip_deg=flip_deg / 2, thickness_mm=SLAB_MM,
                                duration_s=PREP_PULSE_S)
    a_post = -exc.rephaser_area_per_m
    a_pre = float(exc.gz.area) - a_post
    lead = pp.make_trapezoid(channel='z', area=-a_pre, system=opts)
    lead_s = float(pp.calc_duration(lead))
    out = sc.LogicBlock('alpha_over_2')
    out.add(0.0, lead)
    out.add(lead_s, exc(rephase=True))
    return out, lead_s + exc.time_to_center()


def transient_file(path, *, prep, reps=N_REPS):
    '''`reps` repetitions at the centre of k-space after `prep`; returns the first imaging index.'''
    out, at, n, first = sc.LogicBlock('transient'), 0.0, 0, 0
    if prep == 'half-angle':
        block, to_centre = half_angle_prep()
        out.add(0.0, block)
        at = sc.Raster(opts.grad_raster_time).ceil(
            to_centre + TR_S / 2 - kernel.time_to_rf_center())
        n = 1          # the alpha/2 pulse takes the first slot in the phase alternation
    elif isinstance(prep, tuple):
        kind, count = prep
        reps_for = ramp_kernels(count) if kind == 'ramp' else [kernel] * count
        for rep in reps_for:
            out.add(at, rep(line=CENTER_LINE, partition=CENTER_PARTITION, phase_deg=PHASE_STEP * n))
            out.add(at, NAV)
            at, n, first = at + rep.tr_s, n + 1, first + 1
    for _ in range(reps):
        out.add(at, kernel(line=CENTER_LINE, partition=CENTER_PARTITION, phase_deg=PHASE_STEP * n))
        out.add(at, NAV)
        at, n = at + TR_S, n + 1
    sc.compile(out, opts, name=path.stem).write(str(path))
    return first


CANDIDATES = {
    'none':                  ('none', 'transient_none'),
    '10 full-flip dummies':  (('dummy', 10), 'transient_dummy10'),
    'alpha/2 - TR/2':        ('half-angle', 'transient_alpha_over_2'),
    '16-step LFA ramp':      (('ramp', 16), 'transient_lfa16'),
}
FIRST = {}
for name, (prep, stem) in CANDIDATES.items():
    FIRST[name] = transient_file(DIAG_DIR / f'{stem}.seq', prep=prep)
    print(f'{name:22s} first imaging repetition at index {FIRST[name]}')

In [ ]:
#: `curve[candidate][tissue]` -- |signal| at the echo of every acquired repetition.
curve = {
    name: {tissue: np.abs(simulate(DIAG_DIR / f'{stem}.seq', one_voxel(**t))[:, ECHO])
           for tissue, t in TISSUES.items()}
    for name, (_, stem) in CANDIDATES.items()
}
#: Trimmed to begin at the first *imaging* repetition, so the four are compared from one instant.
imaging = {name: {t: s[FIRST[name]:] for t, s in d.items()} for name, d in curve.items()}
print('simulated', len(CANDIDATES) * len(TISSUES), 'single-voxel trains')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), sharex=True)
for ax, tissue in zip(axes, TISSUES):
    for name in CANDIDATES:
        s = imaging[name][tissue]
        ax.plot(s / s[-50:].mean(), lw=1.2, label=name)
    ax.axhline(1.0, color='k', lw=0.6, ls=':')
    ax.set(title=tissue, xlabel='imaging repetition', ylim=(0, 4))
axes[0].set_ylabel('|signal| / eventual')
axes[0].legend(fontsize=7, frameon=False)
fig.suptitle('the opening transient, from the first full-flip repetition', y=1.03)
fig.tight_layout()

print(f'{"preparation":24s}{"first rep / eventual":>34s}{"":>4}{"reps to stay within 5%":>26s}')
print(f'{"":24s}' + ''.join(f'{t:>12s}' for t in TISSUES) + '    ' +
      ''.join(f'{t:>9s}' for t in TISSUES))
for name in CANDIDATES:
    first_cells, settle_cells = [], []
    for tissue in TISSUES:
        s = imaging[name][tissue]
        eventual = s[-50:].mean()
        first_cells.append(f'{s[0] / eventual:>12.2f}')
        bad = np.where(np.abs(s - eventual) / eventual > 0.05)[0]
        settle_cells.append(f'{(bad[-1] + 1 if len(bad) else 0):>9d}')
    print(f'{name:24s}' + ''.join(first_cells) + '    ' + ''.join(settle_cells))

### What that says, and what it does not

Two columns, two different quantities.

**The preparation changes the transient amplitude.** Unprepared, the first imaging repetition is
several times the eventual signal -- it is the free-induction response of fully relaxed
magnetisation, not a steady-state sample. `alpha/2 - TR/2` and a 16-step flip ramp both cut that
roughly in half, and the ten full-flip dummies that this example used to ship sit between them and
nothing.

**The preparation does not change how long convergence takes.** The right-hand column is almost
identical across all four schemes: the decay rate toward the steady state is a property of T1, T2,
TR and the flip angle, and a catalyzing pulse does not alter it. What it alters is where the train
*starts* on that decay.

So the correct reading is "the first repetitions are less wrong", not "the magnetisation is now in
steady state". CSF is the clearest case: it has the smallest transient relative to its own eventual
signal and the longest settling time of the three.

`01` ships **alpha/2 - TR/2**: it ties the best ramp on transient amplitude while costing one pulse
and half a TR instead of sixteen repetitions -- which is what makes it affordable once per segment.

## 2. What did `01` actually write?

Before simulating anything, read the two files back and confirm what acquisition they describe.
This is the step that catches a notebook simulating something other than what it ships.

In [ ]:
def readback(path):
    '''Repetition count, label addresses and declared definitions of a written .seq.'''
    seq = pp.Sequence()
    seq.read(str(path))
    addresses, adcs = [], 0
    for i in range(1, len(seq.block_events) + 1):
        block = seq.get_block(i)
        if getattr(block, 'adc', None) is not None:
            adcs += 1
        label = getattr(block, 'label', None)
        if label is not None:
            got = {lab.label: int(lab.value) for lab in np.atleast_1d(label)}
            if 'LIN' in got and 'PAR' in got:
                addresses.append((got['LIN'], got['PAR']))
    return seq, adcs, addresses


for stem in ('bssfp_3d', 'bssfp_3d_segmented'):
    seq, adcs, addresses = readback(SEQ_DIR / f'{stem}.seq')
    assert addresses == TABLE, f'{stem} does not walk the table 01 recorded'
    assert len(set(addresses)) == len(addresses), f'{stem} writes a k-space address twice'
    print(f'{stem:20s} {adcs:4d} readouts, {len(set(addresses))} distinct (LIN, PAR) addresses, '
          f'{seq.duration()[0]:5.2f} s')
print()
print(f'both files walk the same table in the same order, and it is the table 01 recorded')

In [ ]:
order = np.array(TABLE)
per_segment = NY * PARTITIONS_PER_SEGMENT
centre_at = TABLE.index((CENTER_LINE, CENTER_PARTITION))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.4))
axes[0].scatter(order[:, 0], order[:, 1], c=np.arange(len(order)), cmap='viridis', s=14)
axes[0].scatter([CENTER_LINE], [CENTER_PARTITION], facecolors='none', edgecolors='crimson',
                s=160, lw=1.8, label='centre of k-space')
axes[0].set(xlabel='ky index (line)', ylabel='kz index (partition)',
            title='view order, coloured by acquisition time')
axes[0].legend(fontsize=7, frameon=False)

segment_of = np.arange(len(order)) // per_segment
axes[1].scatter(np.arange(len(order)) % per_segment, segment_of, c=segment_of, cmap='tab10', s=10)
axes[1].axvline(centre_at % per_segment, color='crimson', lw=1.4, ls='--')
axes[1].set(xlabel='repetition within its segment', ylabel='segment',
            title='where each view falls after its own preparation')
fig.tight_layout()

print(f'centre of k-space is repetition {centre_at} of {len(TABLE)}')
print(f'  -> segment {centre_at // per_segment}, {centre_at % per_segment} repetitions after '
      f'that segment\'s preparation')
print(f'each segment is {per_segment} repetitions = {per_segment * TR_S:.2f} s')

**This is the design decision 3D forces that 2D does not.** In a 2D acquisition the transient maps
onto `ky` alone. Here the view order is a path through a `(ky, kz)` plane, and the plot on the left
shows which part of that plane is visited while the magnetisation is still settling.

With partition-major ordering and segments cut by partition, the first repetitions of **every**
segment land on one partition at a time -- so the transient is imprinted along `ky` at four
particular `kz` values rather than smeared over the plane. The right-hand panel is the same fact
seen per segment: the dashed line is where the centre of k-space falls after its own preparation.

Whether that matters is section 4.

## 3. Does the continuous acquisition reconstruct correctly?

A 3D FFT of the sorted table. The three matrix dimensions are all different, so a `ky`/`kz` swap or
a transposed reconstruction changes the **shape** of the volume and cannot be mistaken for a
plausible image.

In [ ]:
def reconstruct(rows):
    '''Sort repetitions into (kx, ky, kz) and take the 3D FFT.

    The table's indices are zero-based with DC at `matrix // 2`, which is exactly the order
    `ifftshift` expects -- so the centre convention the modules use is the one this assumes, and
    nothing is re-derived here.
    '''
    cube = np.zeros((NX, NY, NZ), dtype=complex)
    for row, (line, partition) in enumerate(TABLE):
        cube[:, line, partition] = rows[row]
    return cube, np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(cube)))


def three_planes(volume, title):
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.6))
    views = (('axial  (x, y)', volume[:, :, NZ // 2].T),
             ('coronal (x, z)', volume[:, NY // 2, :].T),
             ('sagittal (y, z)', volume[NX // 2, :, :].T))
    for ax, (name, plane) in zip(axes, views):
        ax.imshow(plane, cmap='gray', origin='lower', aspect='auto')
        ax.set(title=name, xticks=[], yticks=[])
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


if BRAIN_IMAGES:
    brain = ph.built(matrix=NX, nz=NZ, n_coils=1)
    rows_cont = simulate(SEQ_DIR / 'bssfp_3d.seq', brain)
    k_cont, vol_cont = reconstruct(rows_cont)
    three_planes(np.abs(vol_cont), 'continuous 3D bSSFP, reconstructed with a 3D FFT')
    print(f'volume {vol_cont.shape}, peak at '
          f'{np.unravel_index(np.argmax(np.abs(vol_cont)), vol_cont.shape)}')
else:
    print('BRAIN_IMAGES is off -- skipping the two full acquisitions')

The volume is `(48, 32, 8)`: readout, phase encode, partition. The axial plane is the one with the
most detail, the sagittal plane is `32 x 8` and visibly coarse along `z` -- which is what a
15 mm partition over an 80 mm slab looks like, and is the geometry the protocol asked for rather
than an artifact. A transposed reconstruction would make these shapes disagree with the matrix.

## 4. What does interrupted segmentation change?

The segmented file covers the same table in the same order. The only difference is that it is cut
into four prepared blocks with a one-second recovery between them -- which is what an acquisition
does when it has to fit inside a breath-hold or a cardiac window.

So every difference below is a *magnetisation history* effect, not an encoding one.

The modulation is measured on a **point object** first, because a small enough object has
essentially flat k-space across the table -- so whatever structure appears in its *measured*
k-space is the acquisition's own signal modulation, with no anatomy for it to hide behind.

"Small enough" is a real condition rather than a figure of speech, and getting it wrong makes the
measurement meaningless. A box of side `d` has its first k-space null at `1/d`; the table here
reaches `|ky| = 100` 1/m, so a 1 mm box nulls at 1000 1/m and is flat across it, while the 100 mm
box used for the single-location transients above nulls at 10 1/m -- *inside* the table, where the
map would show the phantom instead of the acquisition.

In [ ]:
#: A *small* box, not `one_voxel`'s 100 mm one.  The modulation map below is this object's
#: measured k-space, so the object's own k-space has to be flat across the table: a 1 mm box has
#: its first null at 1000 1/m against a |ky| extent of 100 1/m, where a 100 mm box would put its
#: null inside the table and the map would show the phantom rather than the acquisition.
point = mr0.CustomVoxelPhantom(
    pos=[[0.0, 0.0, 0.0]], PD=1.0, D=0.0, B0=0.0, voxel_size=0.001, voxel_shape='box',
    **TISSUES['white matter'],
).build()
rows_point_cont = simulate(SEQ_DIR / 'bssfp_3d.seq', point)
rows_point_seg = simulate(SEQ_DIR / 'bssfp_3d_segmented.seq', point)

k_point_cont, _ = reconstruct(rows_point_cont)
k_point_seg, _ = reconstruct(rows_point_seg)
#: |k| at the echo sample, as a (ky, kz) map: the acquisition's modulation, with no object in it.
mod_cont = np.abs(k_point_cont[ECHO])
mod_seg = np.abs(k_point_seg[ECHO])
flat = mod_cont[CENTER_LINE, CENTER_PARTITION]

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.2))
for ax, (name, m) in zip(axes, (('continuous', mod_cont), ('segmented', mod_seg))):
    im = ax.imshow(m.T / flat, origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=2.2)
    ax.set(title=f'{name}: |signal| over (ky, kz)', xlabel='ky index', ylabel='kz index')
    fig.colorbar(im, ax=ax, fraction=0.046)
axes[2].plot(mod_cont[:, CENTER_PARTITION] / flat, lw=1.3, label='continuous')
axes[2].plot(mod_seg[:, CENTER_PARTITION] / flat, lw=1.3, label='segmented')
axes[2].axvline(CENTER_LINE, color='crimson', ls=':', lw=1.2)
axes[2].set(title=f'through the centre partition (kz index {CENTER_PARTITION})',
            xlabel='ky index', ylabel='|signal| / centre')
axes[2].legend(fontsize=7, frameon=False)
fig.tight_layout()

for name, m in (('continuous', mod_cont), ('segmented', mod_seg)):
    print(f'{name:12s} range {m.min() / flat:.2f} .. {m.max() / flat:.2f} of the centre sample')
print()
print('mean |signal| per partition, in acquisition order:')
for name, m in (('continuous', mod_cont), ('segmented', mod_seg)):
    print(f'  {name:12s}' + ' '.join(f'{m[:, p].mean() / flat:6.2f}' for p in range(NZ)))
print(f'  {"":12s}' + ' '.join(f'{"|" if p % PARTITIONS_PER_SEGMENT == 0 else "":>6s}'
                               for p in range(NZ)) + '   <- segment boundaries')

Read the per-partition row. Both acquisitions are modulated, but not in the same *shape*, and the
shape is what the reconstruction sees.

**Continuous pays the transient once.** With partition-major ordering that single decay is spread
across the whole `kz` axis, so the partition means fall monotonically from about 1.6 to about 0.9 --
one long ramp from one edge of k-space to the other.

**Segmented pays it four times**, and the segment boundaries sit at fixed partitions. After the
first segment the means settle into a repeating pair, roughly 1.44 then 1.23: a small ripple whose
period is the segment size, riding on a baseline that no longer slopes.

That difference is not the one this section was expected to find. A long monotonic ramp is a large
low-order modulation of k-space, and a small periodic ripple is not -- so the *continuous*
acquisition is the more modulated of the two here, even though it is the one that is never
interrupted. The next cell measures what that does to the point-spread.

In [ ]:
def psf_profile(cube):
    '''The point object's own image: the acquisition's point-spread function.'''
    return np.abs(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(cube))))


psf_cont, psf_seg = psf_profile(k_point_cont), psf_profile(k_point_seg)
z_axis = np.arange(NZ) - NZ // 2

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.2))
for ax, axis_name, profile_of in (
        (axes[0], 'z (partition)', lambda p: p[NX // 2, NY // 2, :]),
        (axes[1], 'y (phase encode)', lambda p: p[NX // 2, :, NZ // 2])):
    a, b = profile_of(psf_cont), profile_of(psf_seg)
    ax.plot(a / a.max(), 'o-', lw=1.2, ms=3, label='continuous')
    ax.plot(b / b.max(), 'o-', lw=1.2, ms=3, label='segmented')
    ax.set(title=f'point-spread along {axis_name}', yscale='log', ylim=(1e-3, 1.5))
    ax.legend(fontsize=7, frameon=False)
fig.tight_layout()

for name, p in (('continuous', psf_cont), ('segmented', psf_seg)):
    z_line = p[NX // 2, NY // 2, :] / p[NX // 2, NY // 2, NZ // 2]
    neighbours = z_line[NZ // 2 - 1] + z_line[NZ // 2 + 1]
    far = z_line[:NZ // 2 - 1].sum() + z_line[NZ // 2 + 2:].sum()
    print(f'{name:12s} total off-peak {z_line.sum() - 1:.3f}   '
          f'immediate neighbours {neighbours:.3f}   further out {far:.3f}')

In [ ]:
if BRAIN_IMAGES:
    rows_seg = simulate(SEQ_DIR / 'bssfp_3d_segmented.seq', brain)
    k_seg, vol_seg = reconstruct(rows_seg)
    three_planes(np.abs(vol_seg), 'segmented 3D bSSFP, four prepared segments')

    a, b = np.abs(vol_cont), np.abs(vol_seg)
    diff = b - a
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.6))
    scale = np.abs(diff).max()
    views = (('axial', diff[:, :, NZ // 2].T), ('coronal', diff[:, NY // 2, :].T),
             ('sagittal', diff[NX // 2, :, :].T))
    for ax, (name, plane) in zip(axes, views):
        im = ax.imshow(plane, cmap='RdBu_r', origin='lower', aspect='auto',
                       vmin=-scale, vmax=scale)
        ax.set(title=f'{name}: segmented - continuous', xticks=[], yticks=[])
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    print(f'peak |difference| {scale:.4g}, {scale / a.max():.3f} of the continuous peak')
    print(f'rms difference    {np.sqrt((diff ** 2).mean()):.4g}')
else:
    print('BRAIN_IMAGES is off -- the point-object modulation above is the quantitative result')

### The interruption is not the thing that hurts here

The continuous acquisition leaks about **three times** as much energy out of the point's own voxel
along `z`, and it does so both into the immediate neighbours and further out. That is the long
monotonic ramp showing up as a point-spread: a smooth slope across `kz` is exactly the kind of
k-space modulation that costs resolution.

**This is worth stating plainly because it is the opposite of the obvious expectation.** The
interrupted acquisition restarts its transient four times and still produces the flatter k-space,
because four short ramps that each start from a similar place vary less across the axis than one
ramp that runs the whole length of it.

**It is also specific to this protocol, and the ratio is what makes it so.** The transient here
decays over roughly 170 repetitions while a segment is 64, so no segment gets far down its own
decay and the four of them look alike. Lengthen the segments, or shorten the transient, and the
segmented acquisition would settle within each segment and the boundaries would become steps
instead -- which is the regime where interruption does cost an artifact. The measurement above is
evidence about this protocol, not a general ordering of the two schemes.

## 5. Does it behave like bSSFP?

Balance and geometry would look the same for any 3D Cartesian acquisition with zero net area. The
signature that says *balanced SSFP* is the off-resonance response: the signal is periodic in
off-resonance with period `1/TR`, with nulls where the per-TR phase reaches an odd multiple of
`pi` relative to the RF phase cycle. Those nulls are the dark **bands**.

A single voxel is enough to show it, swept across one period.

In [ ]:
#: A shorter train than section 1's: the sweep runs it once per offset, and only the settled tail
#: is read, so the repetitions beyond that are cost without information.
BAND_PATH = DIAG_DIR / 'band_sweep.seq'
transient_file(BAND_PATH, prep='half-angle', reps=120)

offsets_hz = np.linspace(-1.0 / TR_S, 1.0 / TR_S, 21)
response = []
for df in offsets_hz:
    rows = simulate(BAND_PATH, one_voxel(B0=float(df), **TISSUES['white matter']))
    response.append(np.abs(rows[-30:, ECHO]).mean())       # the settled part of the train
response = np.array(response)

fig, ax = plt.subplots(figsize=(7.5, 3.0))
ax.plot(offsets_hz, response / response.max(), 'o-', lw=1.3, ms=3)
for null in (-1 / (2 * TR_S), 1 / (2 * TR_S)):
    ax.axvline(null, color='crimson', ls=':', lw=1.2)
ax.set(xlabel='off-resonance (Hz)', ylabel='settled |signal| / max',
       title=f'bSSFP off-resonance response, 0/pi phase cycle, 1/TR = {1 / TR_S:.0f} Hz')
fig.tight_layout()

nulls = offsets_hz[np.argsort(response)[:2]]
print(f'deepest two samples at {nulls[0]:+.0f} and {nulls[1]:+.0f} Hz')
print(f'predicted nulls at {-1 / (2 * TR_S):+.0f} and {1 / (2 * TR_S):+.0f} Hz '
      f'(odd multiples of pi per TR, given the 180 degree RF phase step)')
print(f'passband ripple: {response.max() / response[np.abs(offsets_hz) < 20].min():.2f}x')

With a 180-degree RF phase step the nulls sit at `+-1/(2 TR)` rather than at zero -- the phase cycle
moves the response by half a period, which is exactly why bSSFP is run phase-cycled: it puts the
passband on resonance instead of a null. A different phase increment moves the bands somewhere
else, and `01` leaves that increment to the caller for that reason.

This is the check that distinguishes "balanced gradients" from "balanced SSFP".

## 6. What remains a protocol choice

Everything measured above is a consequence of decisions that live **above** the repetition, and
this notebook has not shown that any of them is optimal -- only what each one does on this protocol.

```text
preparation          alpha/2 - TR/2 here.  It reduces the transient amplitude; it does not
                     shorten convergence, which stays a property of T1, T2, TR and flip angle.

segment size         four segments of two partitions.  Fewer, longer segments pay the transient
                     fewer times; more, shorter segments fit a tighter breath-hold or window.

view order           partition-major with linear ky.  This puts the segment boundaries at fixed
                     kz, which is why the modulation above is banded along the partition axis.
                     Interleaving partitions across segments would spread it; centric ky would
                     put the worst of it on the k-space centre instead of the edge.

recovery interval    one second.  Longer recovery means more relaxation, so a larger transient
                     at the start of the next segment -- the two are not independent.

phase increment      180 degrees, which places the passband on resonance.
```

**Limitations.** One phantom, one field strength, no B1 inhomogeneity, no motion, and a protocol
sized so that its own simulation is affordable rather than sized for a scanner. The off-resonance
sweep is a single voxel, so it shows the response and not what banding does to an image.

## Summary

- The **preparation** was chosen by measuring four candidates on this protocol, holding the
  encoding at the centre of k-space so only magnetisation history differed. `alpha/2 - TR/2`
  reaches the lowest opening transient of the cheap schemes and costs one pulse and half a TR.
- A preparation changes the transient **amplitude**, not the convergence **timescale**. The
  repetitions needed to settle within 5% were nearly identical for all four.
- Both acquisitions were read back from `seq/` and confirmed to walk the table `01` recorded, with
  distinct `(LIN, PAR)` addresses, before anything was simulated.
- The continuous acquisition reconstructs into the expected `48 x 32 x 8` volume in all three
  planes, with the matrix dimensions distinct enough that a swap would change the shape.
- Segmentation pays the transient once per segment. On a point object -- where the acquisition's
  own modulation cannot hide behind anatomy -- that appears as banding along the partition axis,
  because the segment boundaries sit at fixed `kz`.
- The off-resonance response is periodic in `1/TR` with nulls at `+-1/(2 TR)` under the 0/pi phase
  cycle, which is the signature that says balanced SSFP rather than merely balanced gradients.

## References

- Bieri O, Scheffler K. *Fundamentals of balanced steady state free precession MRI.*
  J Magn Reson Imaging 2013;38:2-11.
- Bernstein MA, King KF, Zhou XJ. *Handbook of MRI Pulse Sequences.* Elsevier, 2004. Section 14.1,
  including the alpha/2 - TR/2 catalyzation and the interruption of an established steady state.
- Hargreaves BA, Vasanawala SS, Pauly JM, Nishimura DG. *Characterization and reduction of the
  transient response in steady-state MR imaging.* Magn Reson Med 2001;46:149-158.
- Le Roux P. *Simplified model and stabilization of SSFP sequences.* J Magn Reson 2003;163:23-37.
- Scheffler K, Lehnhardt S. *Principles and applications of balanced SSFP techniques.*
  Eur Radiol 2003;13:2409-2418.